# Molecular docking -- Lamium_amplexicaule vs PPAR-alpha (1K7L)

Protein: Human PPAR-alpha ligand-binding domain, PDB 1K7L, native agonist GW409544 (ligand code 544). Only chain A (the receptor) is used for docking -- the crystal also contains a short co-activator peptide (chain B) bound to the receptor surface, a real part of the active conformation but not the small-molecule ligand pocket, excluded here as is standard practice for nuclear-receptor docking. Compounds: Chrysoeriol, Stigmasterol (as specified by the user; SMILES cross-checked against PubChem canonical structures).

Note: these two compounds differ substantially in size/class from each other and from previous terpene screens in this project -- Chrysoeriol is a flavone (C16H12O6, ~300 g/mol) and Stigmasterol is a large phytosterol (C29H48O, ~413 g/mol, steroid nucleus + flexible side chain). Vina scores across such different chemotypes are not directly comparable to each other in a simple "more negative = more potent" sense; larger, more flexible molecules can pick up more favorable score from sheer surface-area/contact count even without high affinity.

## Content of this notebook

1. Feching system and cleanup
2. Protein and ligand sanitization
3. System Visualization 
4. Docking with AutoDock Vina
    - Receptor preparation
    - Ligand preparation
    - Docking box definition
    - Docking
    - PDBQT results file conversion to SDF
    - 3D visualization of docking results
    - 2D interaction table and map
5. Docking with Smina
    - Receptor preparation
    - Ligand preparation
    - Docking box definition
    - Docking
    - 3D visualization of docking results
    - 2D interaction table and map
6. Docking with LeDock
    - Receptor preparation
    - Ligand preparation
    - Docking box definition
    - Docking
    - DOK results file conversion to SDF
    - 3D visualization of docking results
    - 2D interaction table and map

In [1]:
from pymol import cmd
import py3Dmol

from vina import Vina

from openbabel import pybel

from rdkit import Chem
from rdkit.Chem import AllChem, Draw

from meeko import MoleculePreparation

import MDAnalysis as mda
from MDAnalysis.coordinates import PDB

import prolif as plf
from prolif.plotting.network import LigNetwork


import sys, os
sys.path.insert(1, "/home/ali/medical plants/docking/Jupyter_Dock-main/utilities/")
from utils import fix_protein, getbox, generate_ledock_file, pdbqt_to_sdf, dok_to_sdf


import warnings
warnings.filterwarnings("ignore")
%config Completer.use_jedi = False

/home/ali/Downloads/or/envs/jupdock_clean/lib/python3.10/site-packages/MDAnalysis/topology/tables.py:52: DeprecationWarning: Deprecated in version 2.8.0
MDAnalysis.topology.tables has been moved to MDAnalysis.guesser.tables. This import point will be removed in MDAnalysis version 3.0.0
  warnings.warn(wmsg, category=DeprecationWarning)


It's a good idea to run the test protocols before attempting custom projects. Later, the user can specify the location of his/her project and save all of the files separately.

In [2]:
os.chdir('/home/ali/Desktop/Algerian_Lamiaceae_v2/results/docking_results/')


## 1. Feching system and cleanup

Implementing Pymol is a simple way to download PDB structures. The user can launch this or any other Jupyter Dock's protocol by providing his or her own files.

In [3]:
cmd.fetch(code='1K7L',type='pdb1')

# 1K7L contains the PPAR-alpha LBD (chain A) PLUS a short co-activator
# peptide (chain B) bound to the receptor surface (part of the real active
# conformation, but not the small-molecule ligand pocket). Keep chain A
# only for the docking receptor.
cmd.select(name='Prot', selection='polymer.protein and chain A')
cmd.save(filename='1K7L_clean.pdb', format='pdb', selection='Prot', state=1)

# Native ligand: GW409544, PDB ligand code 544, chain A.
cmd.select(name='NativeLig', selection='resn 544 and chain A')
_natlig_model = cmd.get_model('NativeLig', state=1)
print("Native ligand (GW409544) atom count:", len(_natlig_model.atom))
cmd.save(filename='1K7L_native_lig.mol2', format='mol2', selection='NativeLig', state=1)
cmd.delete('all')


Native ligand (GW409544) atom count: 38


## 2. Protein and ligand sanitization

### 2.1. Protein Sanitization

#### Method 1: LePro

The Lephar molecular docking suite includes a very powerful tool for automatically preparing proteins for molecular docking. As a result, for these protocols, it will be the preferred tool for protein preparation. The protein prepared by LePro can be used in AutoDock VIna and LeDock.

In [4]:
!"/home/ali/medical plants/docking/Jupyter_Dock-main/bin/lepro_linux_x86" # Launch this cell to see parameters


************************************************************
*      LePro                                               *
*            Add hydrogen atoms to a protein &             *
*            write the input file for LeDock               *
*      Copyright (C) 2013-21 Hongtao Zhao, PhD             *
*      Email: htzhaovv@gmail.com                           *
************************************************************
----------Usage:                                                                       
          lepro [PDB file] [-rot || -metal || -p]                                        
          -rot  [[chain] resid] align principal axes of the binding site with Cartesian
          -metal keep ZN/MN/CA/MG                                                      
          -metal -p redistribute metal charge to protein                               



In [5]:
!"/home/ali/medical plants/docking/Jupyter_Dock-main/bin/lepro_linux_x86" {'1K7L_clean.pdb'}

os.rename('pro.pdb','1K7L_clean_H.pdb')


#### Method 2: fix_protein (PDBFixer)

For proteins with missing amino acids or residues, or to ensure a more thorough sanitization of protein Jupyter Dock includes the _fix_protein()_ function, which employs PDBFixer to correct a wide range of common errors in protein pdb files. Furthermore, PDBFixer enables the assignment of pH-dependent protonation states to proteins.


>**Warning 1:**  It is possible to encounter problems with protein fixing when using **_fix_protein()_** and AutoDock Tools' **_prepare_receptor_** or when running LeDock. To resolve the issue, it is best to set the parameter _-A hydrogens_ in _prepare receptor_.


>**Hint:** PDBFixer is a great solution for many systems because it can solve serious problems in PDB files. As a result, PDBFixer renumbers the residues beginning with 1 regardless of the numbering on the original PDB file. To address this issue, the **_fix_protein()_** function includes a protocol for atomically renumbering the residues in accordance with the original PDB file.


**_fix_protein ( params )_**

Params:
      
  - **filename**: _str or path-like_ ; input file containing protein struture to be modified, file extrension must be pdb

  - **addHs_pH**: _float_ ; Add hydrogens at user defined pH

  - **try_renumberResidues**: _bool_ ; By default PDBFixer renumarets residues starting in 1, this option tries to recover originar residues numbering
    
  - **output**: _str or path-like_ ; output filename, extension must be pdb

```
fix_protein(filename='1K7L_clean.pdb',addHs_pH=7.4,try_renumberResidues=True,output='1K7L_clean_H.pdb')
```

## 4. Docking with AutoDock Vina

AutoDock Vina (Vina) is one of the docking engines in the AutoDock Suite, together with AutoDock4 (AD4), AutoDockGPU, AutoDockFR, and AutoDock-CrankPep, and arguably among the most widely used and successful docking engines. The reasons for this success are mostly due to its ease of use and its speed (up to 100x faster than AD4), when compared to the other docking engines in the suite and elsewhere, as well as being open source.

### 4.1. Protein preparation

After sanitization, the protein docking preparation includes converting it to the PDBQT file format, which stores the atomic coordinates, partial charges, and AutoDock atom types for both the receptor and the ligand. 

>**Hint:** Despite the fact that the PDBQT format includes changes (Q) and atom types (T) for molecular docking. The charges are not required for Autodock Vina, which computes electrostatic interactions using its own force field. Although, when using AutoDock instead of AutoDock Vina, the Q term is required. More information can be found here:https://autodock-vina.readthedocs.io/en/latest/introduction.html

#### Method 1: AutoDock Tools prepare_receptor

The AutoDock Tools are the best way to prepare the receptor for AutoDock Vina. Nonetheless, AutoDock Tools is a comprehensive suite of programs and scripts that are difficult to manage. Jupyter Dock execute the **_prepare receptor_** and **_prepare_ligand_** functions on their own. As a result, obtaining proper PDBQT files for AutoDock Vina has never been easier.

>**Warning:** There is currently only one method for preparing a PDBQT file for a receptor. It is expected that new methods apart from AutoDock Tools executables will be included in the near future into Jupyter Dock.

In [6]:
!"/home/ali/medical plants/docking/Jupyter_Dock-main/bin/prepare_receptor" #Lauch this cell to see parameters


prepare_receptor4: receptor filename must be specified.
Usage: prepare_receptor4.py -r filename

    Description of command...
         -r   receptor_filename 
        supported file types include pdb,mol2,pdbq,pdbqs,pdbqt, possibly pqr,cif
    Optional parameters:
        [-v]  verbose output (default is minimal output)
        [-o pdbqt_filename]  (default is 'molecule_name.pdbqt')
        [-A]  type(s) of repairs to make: 
             'bonds_hydrogens': build bonds and add hydrogens 
             'bonds': build a single bond from each atom with no bonds to its closest neighbor
             'hydrogens': add hydrogens
             'checkhydrogens': add hydrogens only if there are none already
             'None': do not make any repairs 
             (default is 'None')
        [-C]  preserve all input charges ie do not add new charges 
             (default is addition of gasteiger charges)
        [-p]  preserve input charges on specific atom types, eg -p Zn -p Fe
        [-U]  cle

In [7]:
!"/home/ali/medical plants/docking/Jupyter_Dock-main/bin/prepare_receptor" -v -r {'1K7L_clean_H.pdb'} -o {'1K7L_clean_H.pdbqt'}

# If this raises "UnboundLocalError: local variable 'bonds' referenced before
# assignment", use this fallback:
# import subprocess
# r = subprocess.run(["obabel", "1K7L_clean_H.pdb", "-O", "1K7L_clean_H.pdbqt", "-xr"],
#                     capture_output=True, text=True)
# print(r.stdout, r.stderr)


set verbose to  True
set receptor_filename to  1K7L_clean_H.pdb
set outputfilename to  1K7L_clean_H.pdbqt
read  1K7L_clean_H.pdb
setting up RPO with mode= automatic and outputfilename=  1K7L_clean_H.pdbqt
charges_to_add= gasteiger
delete_single_nonstd_residues= None
adding gasteiger charges to peptide


In [8]:
fix_protein(filename='1K7L_clean.pdb',addHs_pH=7.4,try_renumberResidues=True,output='1K7L_clean_H_fix.pdb')

## 7. Screening PHYTO compounds (Lamium_amplexicaule) against PPAR-alpha (1K7L)

Reuses the receptor (`1K7L_clean_H.pdb`/`.pdbqt`, chain A only) and the
docking box (native ligand GW409544, chain A, `state=1`) prepared above.
Each compound docked in its own cell with its own uniquely-named
variables, seed pinned at 42. Compounds: Chrysoeriol, Stigmasterol
(as specified by the user).

Note on ligand flexibility: Stigmasterol has 5 rotatable bonds in its
side chain, several more than the small monoterpenes docked in earlier
notebooks in this project -- `n_confs=20` conformer sampling and Vina's
own internal search should handle this, but if pose quality looks poor,
consider raising `exhaustiveness` for this compound specifically.

Reference from the PPAR-alpha literature: the real synthetic PPARalpha
drug fenofibrate scores -6.24 kcal/mol in a comparable docking study.

In [9]:
import os
from rdkit import Chem
from rdkit.Chem import AllChem, rdMolDescriptors
from openbabel import openbabel as ob
from vina import Vina
import pandas as pd
import matplotlib.pyplot as plt

DOCKING_DIR = "/home/ali/Desktop/Algerian_Lamiaceae_v2/results/docking_results/"
FIGURES_DIR = os.path.join(DOCKING_DIR, "figures")
os.makedirs(FIGURES_DIR, exist_ok=True)
os.chdir(DOCKING_DIR)

RECEPTOR_PDB   = os.path.join(DOCKING_DIR, "1K7L_clean_H.pdb")
RECEPTOR_PDBQT = os.path.join(DOCKING_DIR, "1K7L_clean_H.pdbqt")
NATIVE_LIG     = os.path.join(DOCKING_DIR, "1K7L_native_lig.mol2")

assert os.path.exists(RECEPTOR_PDBQT), "Run the receptor prep cells above first."
assert os.path.exists(NATIVE_LIG), "Run the native ligand extraction cell above first."

VINA_SEED = 42  # pinned for reproducibility

def prepare_ligand_from_smiles(name, smiles, n_confs=20):
    """SMILES -> multiple 3D conformers (ETKDGv3) -> MMFF94 minimize each ->
    keep lowest-energy conformer -> PDB -> PDBQT (AutoDockTools prepare_ligand,
    with OpenBabel fallback)."""
    import subprocess
    m = Chem.MolFromSmiles(smiles)
    m = Chem.AddHs(m)

    params = AllChem.ETKDGv3()
    params.randomSeed = 42
    conf_ids = AllChem.EmbedMultipleConfs(m, numConfs=n_confs, params=params)

    energies = []
    for cid in conf_ids:
        ff = AllChem.MMFFGetMoleculeForceField(m, AllChem.MMFFGetMoleculeProperties(m), confId=cid)
        if ff is None:
            continue
        ff.Minimize(maxIts=2000)
        energies.append((ff.CalcEnergy(), cid))
    if not energies:
        for cid in conf_ids:
            ff = AllChem.UFFGetMoleculeForceField(m, confId=cid)
            ff.Minimize(maxIts=2000)
            energies.append((ff.CalcEnergy(), cid))

    best_energy, best_cid = min(energies, key=lambda x: x[0])
    print(f"  {name}: {len(conf_ids)} conformers generated, "
          f"lowest energy = {best_energy:.2f} kcal/mol (conf {best_cid})")

    pdb_path   = os.path.join(DOCKING_DIR, f'_lig_{name}.pdb')
    pdbqt_path = os.path.join(DOCKING_DIR, f'_lig_{name}.pdbqt')
    Chem.MolToPDBFile(m, pdb_path, confId=best_cid)

    PREP_LIG_BIN = "/home/ali/medical plants/docking/Jupyter_Dock-main/bin/prepare_ligand"
    result = subprocess.run([PREP_LIG_BIN, "-v", "-l", pdb_path, "-o", pdbqt_path],
                             capture_output=True, text=True)
    if not os.path.exists(pdbqt_path) or os.path.getsize(pdbqt_path) == 0:
        obConv = ob.OBConversion()
        obConv.SetInAndOutFormats('pdb', 'pdbqt')
        obmol = ob.OBMol()
        obConv.ReadFile(obmol, pdb_path)
        obConv.WriteFile(obmol, pdbqt_path)

    return pdbqt_path


### 7.1. Recompute the docking box (native ligand GW409544, `state=1`)

In [10]:
cmd.load(filename=RECEPTOR_PDB, format='pdb', object='prot', state=1)
cmd.load(filename=NATIVE_LIG, format='mol2', object='natlig', state=1)

center, size = getbox(selection='natlig', extending=5.0, software='vina')
cmd.delete('all')

print(center, '\n', size)


{'center_x': -16.445500373840332, 'center_y': -15.031999588012695, 'center_z': -3.883500039577484} 
 {'size_x': 20.571001052856445, 'size_y': 18.091999053955078, 'size_z': 20.402999997138977}


In [11]:
import subprocess
from rdkit.Chem.rdMolAlign import GetBestRMS

REDOCK_DIR = os.path.join(DOCKING_DIR, "redocking_validation")
os.makedirs(REDOCK_DIR, exist_ok=True)

PDB_CODE = os.path.basename(RECEPTOR_PDB).split('_')[0]

# 1) Prepare the NATIVE ligand for docking -- from its crystal geometry
native_lig_pdbqt = os.path.join(REDOCK_DIR, f"{PDB_CODE}_native_lig_redock.pdbqt")
PREP_LIG_BIN = "/home/ali/medical plants/docking/Jupyter_Dock-main/bin/prepare_ligand"
result = subprocess.run([PREP_LIG_BIN, "-v", "-l", NATIVE_LIG, "-o", native_lig_pdbqt],
                         capture_output=True, text=True)
if not os.path.exists(native_lig_pdbqt) or os.path.getsize(native_lig_pdbqt) == 0:
    from openbabel import openbabel as ob
    obConv = ob.OBConversion()
    obConv.SetInAndOutFormats('mol2', 'pdbqt')
    obmol = ob.OBMol()
    obConv.ReadFile(obmol, NATIVE_LIG)
    obConv.WriteFile(obmol, native_lig_pdbqt)

# 2) Redock into the SAME box already computed for compound screening (7.1)
#    exhaustiveness raised from 10 -> 32: at exhaustiveness=10 the top-2
#    poses were separated by only 0.11 kcal/mol (score noise), so the
#    search was under-sampled for a ligand with this many rotatable bonds.
REDOCK_EXHAUSTIVENESS = 72

v_redock = Vina(sf_name='vina', seed=VINA_SEED)
v_redock.set_receptor(RECEPTOR_PDBQT)
v_redock.set_ligand_from_file(native_lig_pdbqt)
v_redock.compute_vina_maps(center=[center['center_x'], center['center_y'], center['center_z']],
                            box_size=[size['size_x'], size['size_y'], size['size_z']])
v_redock.dock(exhaustiveness=REDOCK_EXHAUSTIVENESS, n_poses=10)

redock_out_pdbqt = os.path.join(REDOCK_DIR, f"{PDB_CODE}_native_lig_redock_out.pdbqt")
v_redock.write_poses(redock_out_pdbqt, n_poses=10, overwrite=True)
redock_energies = v_redock.energies(n_poses=10)
redock_best_score = round(float(redock_energies[0][0]), 3)
print(f"Redocking best Vina score: {redock_best_score} kcal/mol (exhaustiveness={REDOCK_EXHAUSTIVENESS})")

redock_out_sdf = redock_out_pdbqt.replace('.pdbqt', '.sdf')
pdbqt_to_sdf(pdbqt_file=redock_out_pdbqt, output=redock_out_sdf)

# 3) RMSD of each docked pose vs the ORIGINAL crystallographic pose.
#    Bond orders on BOTH sides re-derived from an independent literature
#    template (GW409544, PubChem CID 446642) -- neither the PyMOL mol2
#    export nor the PDBQT->SDF round trip reliably preserves bond orders.
GW409544_TEMPLATE_SMILES = "CC1=C(N=C(O1)C2=CC=CC=C2)CCOC3=CC=C(C=C3)CC(C(=O)O)NC(=CC(=O)C4=CC=CC=C4)C"
template = Chem.MolFromSmiles(GW409544_TEMPLATE_SMILES)

ref_mol_raw = Chem.MolFromMol2File(NATIVE_LIG, sanitize=False, removeHs=True)
try:
    ref_mol = AllChem.AssignBondOrdersFromTemplate(template, ref_mol_raw)
except Exception as e:
    print(f"Could not fix ref_mol bond orders from template: {e}")
    ref_mol = None

poses = Chem.SDMolSupplier(redock_out_sdf, removeHs=True)
rmsd_rows = []
for i, pose_mol in enumerate(poses):
    if pose_mol is None or ref_mol is None:
        rmsd_rows.append({"Pose": i + 1, "Vina_score_kcal_mol": float(redock_energies[i][0]),
                           "RMSD_vs_crystal_pose_A": None})
        continue
    try:
        pose_mol_fixed = AllChem.AssignBondOrdersFromTemplate(template, pose_mol)
        rmsd = round(GetBestRMS(pose_mol_fixed, ref_mol), 3)
    except Exception as e:
        print(f"  Pose {i+1}: RMSD failed ({e}) -- inspect this pose manually")
        rmsd = None
    rmsd_rows.append({
        "Pose": i + 1,
        "Vina_score_kcal_mol": float(redock_energies[i][0]),
        "RMSD_vs_crystal_pose_A": rmsd,
    })

redock_df = pd.DataFrame(rmsd_rows)
print(f"=== Reference-ligand redocking validation: {PDB_CODE} / native ligand {NATIVE_LIG} (exhaustiveness={REDOCK_EXHAUSTIVENESS}) ===")
display(redock_df)

best_pose_rmsd = redock_df.loc[redock_df['Vina_score_kcal_mol'].idxmin(), 'RMSD_vs_crystal_pose_A']
print(f"Best-scoring pose RMSD vs crystal structure: {best_pose_rmsd} Å")
print("Standard success criterion in the docking-validation literature: RMSD < 2.0 A")
if best_pose_rmsd is not None and best_pose_rmsd < 2.0:
    print("Redocking validation result: PASS")
else:
    print("Redocking validation result: FAIL")
    print("  -> RMSD >= 2.0 A (or could not be computed): inspect the pose visually")
    print("     before relying on this receptor/box/protocol for compound screening.")

redock_df.to_csv(os.path.join(REDOCK_DIR, f"{PDB_CODE}_redocking_validation.csv"), index=False)

[RDKit] WARNING:[10:57:48] 1K7L: Warning - no explicit hydrogens in mol2 file but needed for formal charge estimation.
[RDKit] WARNING:[10:57:48] WARNING: More than one matching pattern found - picking one
[RDKit] WARNING:


Computing Vina grid ... done.
Performing docking (random seed: 42) ... 
0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************
Redocking best Vina score: -11.159 kcal/mol (exhaustiveness=72)
=== Reference-ligand redocking validation: 1K7L / native ligand /home/ali/Desktop/Algerian_Lamiaceae_v2/results/docking_results/1K7L_native_lig.mol2 (exhaustiveness=72) ===


,Pose,Vina_score_kcal_mol,RMSD_vs_crystal_pose_A
0,1,-11.159,2.728
1,2,-10.909,0.721
2,3,-10.828,2.303
3,4,-10.735,2.943
4,5,-10.624,3.622
5,6,-10.520,3.890
6,7,-10.515,1.681
7,8,-10.471,1.579
8,9,-10.258,2.532
9,10,-10.072,3.483


Best-scoring pose RMSD vs crystal structure: 2.728 Å
Standard success criterion in the docking-validation literature: RMSD < 2.0 A
Redocking validation result: FAIL
  -> RMSD >= 2.0 A (or could not be computed): inspect the pose visually
     before relying on this receptor/box/protocol for compound screening.


In [12]:
# 3) RMSD of each docked pose vs the ORIGINAL crystallographic pose
#    (heavy atoms only). Bond orders on BOTH sides are re-derived from an
#    independent literature template (GW409544, PubChem CID 446642), since
#    neither the PyMOL-exported mol2 nor the PDBQT->SDF round trip reliably
#    preserves bond orders/aromaticity on their own.

GW409544_TEMPLATE_SMILES = "CC1=C(N=C(O1)C2=CC=CC=C2)CCOC3=CC=C(C=C3)CC(C(=O)O)NC(=CC(=O)C4=CC=CC=C4)C"
template = Chem.MolFromSmiles(GW409544_TEMPLATE_SMILES)

ref_mol_raw = Chem.MolFromMol2File(NATIVE_LIG, sanitize=False, removeHs=True)
try:
    ref_mol = AllChem.AssignBondOrdersFromTemplate(template, ref_mol_raw)
except Exception as e:
    print(f"Could not fix ref_mol bond orders from template: {e}")
    ref_mol = None

poses = Chem.SDMolSupplier(redock_out_sdf, removeHs=True)
rmsd_rows = []
for i, pose_mol in enumerate(poses):
    if pose_mol is None or ref_mol is None:
        rmsd_rows.append({"Pose": i + 1, "Vina_score_kcal_mol": float(redock_energies[i][0]),
                           "RMSD_vs_crystal_pose_A": None})
        continue
    try:
        pose_mol_fixed = AllChem.AssignBondOrdersFromTemplate(template, pose_mol)
        rmsd = round(GetBestRMS(pose_mol_fixed, ref_mol), 3)
    except Exception as e:
        print(f"  Pose {i+1}: RMSD failed ({e}) -- inspect this pose manually")
        rmsd = None
    rmsd_rows.append({
        "Pose": i + 1,
        "Vina_score_kcal_mol": float(redock_energies[i][0]),
        "RMSD_vs_crystal_pose_A": rmsd,
    })

redock_df = pd.DataFrame(rmsd_rows)
print(f"=== Reference-ligand redocking validation: {PDB_CODE} / native ligand {NATIVE_LIG} ===")
display(redock_df)

best_pose_rmsd = redock_df.loc[redock_df['Vina_score_kcal_mol'].idxmin(), 'RMSD_vs_crystal_pose_A']
print(f"Best-scoring pose RMSD vs crystal structure: {best_pose_rmsd} Å")
print("Standard success criterion in the docking-validation literature: RMSD < 2.0 A")
if best_pose_rmsd is not None and best_pose_rmsd < 2.0:
    print("Redocking validation result: PASS")
else:
    print("Redocking validation result: FAIL")
    print("  -> RMSD >= 2.0 A (or could not be computed): inspect the pose visually")
    print("     before relying on this receptor/box/protocol for compound screening.")

redock_df.to_csv(os.path.join(REDOCK_DIR, f"{PDB_CODE}_redocking_validation.csv"), index=False)

[RDKit] WARNING:[10:57:48] 1K7L: Warning - no explicit hydrogens in mol2 file but needed for formal charge estimation.
[RDKit] WARNING:[10:57:48] WARNING: More than one matching pattern found - picking one
[RDKit] WARNING:


=== Reference-ligand redocking validation: 1K7L / native ligand /home/ali/Desktop/Algerian_Lamiaceae_v2/results/docking_results/1K7L_native_lig.mol2 ===


,Pose,Vina_score_kcal_mol,RMSD_vs_crystal_pose_A
0,1,-11.159,2.728
1,2,-10.909,0.721
2,3,-10.828,2.303
3,4,-10.735,2.943
4,5,-10.624,3.622
5,6,-10.520,3.890
6,7,-10.515,1.681
7,8,-10.471,1.579
8,9,-10.258,2.532
9,10,-10.072,3.483


Best-scoring pose RMSD vs crystal structure: 2.728 Å
Standard success criterion in the docking-validation literature: RMSD < 2.0 A
Redocking validation result: FAIL
  -> RMSD >= 2.0 A (or could not be computed): inspect the pose visually
     before relying on this receptor/box/protocol for compound screening.


In [13]:
SEEDS = [1, 7, 42, 99, 123]
replicate_rows = []

for seed in SEEDS:
    v_rep = Vina(sf_name='vina', seed=seed)
    v_rep.set_receptor(RECEPTOR_PDBQT)
    v_rep.set_ligand_from_file(native_lig_pdbqt)
    v_rep.compute_vina_maps(center=[center['center_x'], center['center_y'], center['center_z']],
                             box_size=[size['size_x'], size['size_y'], size['size_z']])
    v_rep.dock(exhaustiveness=REDOCK_EXHAUSTIVENESS, n_poses=10)

    out_pdbqt = os.path.join(REDOCK_DIR, f"{PDB_CODE}_redock_seed{seed}.pdbqt")
    v_rep.write_poses(out_pdbqt, n_poses=10, overwrite=True)
    energies = v_rep.energies(n_poses=10)
    out_sdf = out_pdbqt.replace('.pdbqt', '.sdf')
    pdbqt_to_sdf(pdbqt_file=out_pdbqt, output=out_sdf)

    poses = Chem.SDMolSupplier(out_sdf, removeHs=True)
    for i, pose_mol in enumerate(poses):
        try:
            pose_fixed = AllChem.AssignBondOrdersFromTemplate(template, pose_mol)
            rmsd = round(GetBestRMS(pose_fixed, ref_mol), 3)
        except Exception:
            rmsd = None
        replicate_rows.append({
            "Seed": seed, "Pose": i + 1,
            "Vina_score_kcal_mol": float(energies[i][0]),
            "RMSD_vs_crystal_pose_A": rmsd,
        })

rep_df = pd.DataFrame(replicate_rows)
print("=== Redocking reproducibility across 5 independent seeds ===")
top1_per_seed = rep_df.sort_values("Vina_score_kcal_mol").groupby("Seed").first().reset_index()
display(top1_per_seed[["Seed", "Pose", "Vina_score_kcal_mol", "RMSD_vs_crystal_pose_A"]])

n_pass = (top1_per_seed["RMSD_vs_crystal_pose_A"] < 2.0).sum()
print(f"\nTop-1 pose RMSD < 2.0 A in {n_pass} of {len(SEEDS)} independent runs.")
best_per_seed = rep_df.loc[rep_df.groupby("Seed")["RMSD_vs_crystal_pose_A"].idxmin()]
print(f"Best RMSD found anywhere in top-10, per seed:")
display(best_per_seed[["Seed", "Pose", "Vina_score_kcal_mol", "RMSD_vs_crystal_pose_A"]])

rep_df.to_csv(os.path.join(REDOCK_DIR, f"{PDB_CODE}_redocking_reproducibility.csv"), index=False)

[RDKit] WARNING:[10:59:48] WARNING: More than one matching pattern found - picking one
[RDKit] WARNING:
[RDKit] WARNING:[10:59:48] WARNING: More than one matching pattern found - picking one
[RDKit] WARNING:



mode |   affinity | dist from best mode
     | (kcal/mol) | rmsd l.b.| rmsd u.b.
-----+------------+----------+----------
   1       -11.16          0          0
   2       -10.91      1.731      8.498
   3       -10.83      1.409      2.481
   4       -10.73      2.287      9.678
   5       -10.62      2.886      4.841
   6       -10.52      1.684      3.758
   7       -10.51      2.223      9.241
   8       -10.47      1.758      8.641
   9       -10.26      2.498      3.324
  10       -10.07      3.483      10.91
Computing Vina grid ... done.
Performing docking (random seed: 1) ... 
0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************


[RDKit] WARNING:[11:01:47] WARNING: More than one matching pattern found - picking one
[RDKit] WARNING:



mode |   affinity | dist from best mode
     | (kcal/mol) | rmsd l.b.| rmsd u.b.
-----+------------+----------+----------
   1       -11.12          0          0
   2        -10.9      1.736      8.526
   3       -10.86      1.458      2.493
   4       -10.75      2.252       9.66
   5       -10.64      2.164      9.226
   6       -10.56      2.988          5
   7       -10.46      1.953      4.296
   8       -10.39      3.526      10.99
   9       -10.34      1.725      8.483
  10       -10.15      2.513      3.346
Computing Vina grid ... done.
Performing docking (random seed: 7) ... 
0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************

mode |   affinity | dist from best mode
     | (kcal/mol) | rmsd l.b.| rmsd u.b.
-----+------------+----------+----------
   1       -11.11          0          0
   2       -10.88      1.724      8.465
   3       -10.84      1.418      2.446
   4

[RDKit] WARNING:[11:07:10] WARNING: More than one matching pattern found - picking one
[RDKit] WARNING:



mode |   affinity | dist from best mode
     | (kcal/mol) | rmsd l.b.| rmsd u.b.
-----+------------+----------+----------
   1        -11.1          0          0
   2       -10.91      1.746      8.511
   3       -10.87       1.45      2.505
   4       -10.72      2.242      9.642
   5       -10.62      2.874       4.86
   6       -10.48       2.14      9.183
   7       -10.41      3.548         11
   8       -10.37      1.895      4.182
   9       -10.28      1.832      8.557
  10       -10.24      2.467      3.307
Computing Vina grid ... done.
Performing docking (random seed: 123) ... 
0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************
=== Redocking reproducibility across 5 independent seeds ===


,Seed,Pose,Vina_score_kcal_mol,RMSD_vs_crystal_pose_A
0,1,1,-11.124,2.713
1,7,1,-11.114,2.717
2,42,1,-11.159,2.728
3,99,1,-11.100,2.723
4,123,1,-11.129,2.705



Top-1 pose RMSD < 2.0 A in 0 of 5 independent runs.
Best RMSD found anywhere in top-10, per seed:


,Seed,Pose,Vina_score_kcal_mol,RMSD_vs_crystal_pose_A
1,1,2,-10.905,0.707
11,7,2,-10.882,0.732
21,42,2,-10.909,0.721
31,99,2,-10.909,0.690
41,123,2,-10.906,0.697


### 7.2. Dock **Chrysoeriol** against PPAR-alpha (1K7L)

In [14]:
# ---- Chrysoeriol (Lamium_amplexicaule) ----
smiles_Chrysoeriol = "COC1=C(C=CC(=C1)C2=CC(=O)C3=C(C=C(C=C3O2)O)O)O"
lig_pdbqt_Chrysoeriol = prepare_ligand_from_smiles("Chrysoeriol", smiles_Chrysoeriol)

v_Chrysoeriol = Vina(sf_name='vina', seed=VINA_SEED)
v_Chrysoeriol.set_receptor(RECEPTOR_PDBQT)
v_Chrysoeriol.set_ligand_from_file(lig_pdbqt_Chrysoeriol)
v_Chrysoeriol.compute_vina_maps(center=[center['center_x'], center['center_y'], center['center_z']],
                     box_size=[size['size_x'], size['size_y'], size['size_z']])

v_Chrysoeriol.dock(exhaustiveness=10, n_poses=10)
vina_out_pdbqt_Chrysoeriol = os.path.join(DOCKING_DIR, "Chrysoeriol_1K7L_vina_out.pdbqt")
v_Chrysoeriol.write_poses(vina_out_pdbqt_Chrysoeriol, n_poses=10, overwrite=True)

energies_Chrysoeriol = v_Chrysoeriol.energies(n_poses=10)
best_score_Chrysoeriol = round(float(energies_Chrysoeriol[0][0]), 3)

print("=== Chrysoeriol ===")
print(f"Best Vina score: {best_score_Chrysoeriol} kcal/mol")
for i, e in enumerate(energies_Chrysoeriol):
    print(f"  Pose {i+1}: {e[0]:.3f}")

vina_out_sdf_Chrysoeriol = vina_out_pdbqt_Chrysoeriol.replace('.pdbqt', '.sdf')
pdbqt_to_sdf(pdbqt_file=vina_out_pdbqt_Chrysoeriol, output=vina_out_sdf_Chrysoeriol)


  Chrysoeriol: 20 conformers generated, lowest energy = 31.92 kcal/mol (conf 3)
=== Chrysoeriol ===
Best Vina score: -7.9 kcal/mol
  Pose 1: -7.900
  Pose 2: -7.581
  Pose 3: -7.396
  Pose 4: -7.373
  Pose 5: -7.341
  Pose 6: -7.093
  Pose 7: -6.957
  Pose 8: -6.878
  Pose 9: -6.830

mode |   affinity | dist from best mode
     | (kcal/mol) | rmsd l.b.| rmsd u.b.
-----+------------+----------+----------
   1       -11.13          0          0
   2       -10.91      1.713      8.435
   3       -10.87        1.4      2.445
   4       -10.73      2.231      9.578
   5       -10.64      2.941      4.906
   6       -10.51      2.262      9.237
   7       -10.44      3.562      10.96
   8        -10.3      1.833      8.514
   9       -10.29      3.618      5.759
  10       -10.17      3.516      10.57
Computing Vina grid ... 

done.
Performing docking (random seed: 42) ... 
0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************


### 7.3. Dock **Stigmasterol** against PPAR-alpha (1K7L)

In [ ]:
# ---- Stigmasterol (Lamium_amplexicaule) ----
smiles_Stigmasterol = "CCC(C=CC(C)C1CCC2C1(CCC3C2CC=C4C3(CCC(C4)O)C)C)C(C)C"
lig_pdbqt_Stigmasterol = prepare_ligand_from_smiles("Stigmasterol", smiles_Stigmasterol)

v_Stigmasterol = Vina(sf_name='vina', seed=VINA_SEED)
v_Stigmasterol.set_receptor(RECEPTOR_PDBQT)
v_Stigmasterol.set_ligand_from_file(lig_pdbqt_Stigmasterol)
v_Stigmasterol.compute_vina_maps(center=[center['center_x'], center['center_y'], center['center_z']],
                     box_size=[size['size_x'], size['size_y'], size['size_z']])

v_Stigmasterol.dock(exhaustiveness=10, n_poses=10)
vina_out_pdbqt_Stigmasterol = os.path.join(DOCKING_DIR, "Stigmasterol_1K7L_vina_out.pdbqt")
v_Stigmasterol.write_poses(vina_out_pdbqt_Stigmasterol, n_poses=10, overwrite=True)

energies_Stigmasterol = v_Stigmasterol.energies(n_poses=10)
best_score_Stigmasterol = round(float(energies_Stigmasterol[0][0]), 3)

print("=== Stigmasterol ===")
print(f"Best Vina score: {best_score_Stigmasterol} kcal/mol")
for i, e in enumerate(energies_Stigmasterol):
    print(f"  Pose {i+1}: {e[0]:.3f}")

vina_out_sdf_Stigmasterol = vina_out_pdbqt_Stigmasterol.replace('.pdbqt', '.sdf')
pdbqt_to_sdf(pdbqt_file=vina_out_pdbqt_Stigmasterol, output=vina_out_sdf_Stigmasterol)


### 7.4. Dock **7-O-Glucosyl-3-methylkaempferol** against PPAR-alpha (1K7L)

Flavonoid glycoside isolated from the aerial parts (butanolic extract) of *Lamium amplexicaule* by Laggoune et al. (2008); reported formula C22H22O11, structurally a 3-O-methyl ether of kaempferol-7-O-glucoside (cf. PubChem CID 5480982 for the unmethylated parent, C21H20O11).

In [ ]:
# ---- 7-O-Glucosyl-3-methylkaempferol (Lamium_amplexicaule, Laggoune et al. 2008) ----
smiles_GlcMethylKaempferol = "COc1c(-c2ccc(O)cc2)oc2cc(O[C@@H]3O[C@H](CO)[C@@H](O)[C@H](O)[C@H]3O)cc(O)c2c1=O"
lig_pdbqt_GlcMethylKaempferol = prepare_ligand_from_smiles("GlcMethylKaempferol", smiles_GlcMethylKaempferol)

v_GlcMethylKaempferol = Vina(sf_name='vina', seed=VINA_SEED)
v_GlcMethylKaempferol.set_receptor(RECEPTOR_PDBQT)
v_GlcMethylKaempferol.set_ligand_from_file(lig_pdbqt_GlcMethylKaempferol)
v_GlcMethylKaempferol.compute_vina_maps(center=[center['center_x'], center['center_y'], center['center_z']],
                     box_size=[size['size_x'], size['size_y'], size['size_z']])

v_GlcMethylKaempferol.dock(exhaustiveness=10, n_poses=10)
vina_out_pdbqt_GlcMethylKaempferol = os.path.join(DOCKING_DIR, "GlcMethylKaempferol_1K7L_vina_out.pdbqt")
v_GlcMethylKaempferol.write_poses(vina_out_pdbqt_GlcMethylKaempferol, n_poses=10, overwrite=True)

energies_GlcMethylKaempferol = v_GlcMethylKaempferol.energies(n_poses=10)
best_score_GlcMethylKaempferol = round(float(energies_GlcMethylKaempferol[0][0]), 3)

print("=== GlcMethylKaempferol ===")
print(f"Best Vina score: {best_score_GlcMethylKaempferol} kcal/mol")
for i, e in enumerate(energies_GlcMethylKaempferol):
    print(f"  Pose {i+1}: {e[0]:.3f}")

vina_out_sdf_GlcMethylKaempferol = vina_out_pdbqt_GlcMethylKaempferol.replace('.pdbqt', '.sdf')
pdbqt_to_sdf(pdbqt_file=vina_out_pdbqt_GlcMethylKaempferol, output=vina_out_sdf_GlcMethylKaempferol)


### 7.5. Dock **Sitosterol** against PPAR-alpha (1K7L)

Phytosterol isolated from the roots (Soxhlet acetone extraction) of *Lamium amplexicaule* by Laggoune et al. (2008); C29H50O, structurally identical to stigmasterol above except for a saturated C22-C23 side-chain bond.

In [ ]:
# ---- Sitosterol (Lamium_amplexicaule, Laggoune et al. 2008) ----
smiles_Sitosterol = "CC[C@H](CC[C@@H](C)[C@H]1CC[C@@H]2[C@@]1(CC[C@H]3[C@H]2CC=C4[C@@]3(CC[C@@H](C4)O)C)C)C(C)C"
lig_pdbqt_Sitosterol = prepare_ligand_from_smiles("Sitosterol", smiles_Sitosterol)

v_Sitosterol = Vina(sf_name='vina', seed=VINA_SEED)
v_Sitosterol.set_receptor(RECEPTOR_PDBQT)
v_Sitosterol.set_ligand_from_file(lig_pdbqt_Sitosterol)
v_Sitosterol.compute_vina_maps(center=[center['center_x'], center['center_y'], center['center_z']],
                     box_size=[size['size_x'], size['size_y'], size['size_z']])

v_Sitosterol.dock(exhaustiveness=10, n_poses=10)
vina_out_pdbqt_Sitosterol = os.path.join(DOCKING_DIR, "Sitosterol_1K7L_vina_out.pdbqt")
v_Sitosterol.write_poses(vina_out_pdbqt_Sitosterol, n_poses=10, overwrite=True)

energies_Sitosterol = v_Sitosterol.energies(n_poses=10)
best_score_Sitosterol = round(float(energies_Sitosterol[0][0]), 3)

print("=== Sitosterol ===")
print(f"Best Vina score: {best_score_Sitosterol} kcal/mol")
for i, e in enumerate(energies_Sitosterol):
    print(f"  Pose {i+1}: {e[0]:.3f}")

vina_out_sdf_Sitosterol = vina_out_pdbqt_Sitosterol.replace('.pdbqt', '.sdf')
pdbqt_to_sdf(pdbqt_file=vina_out_pdbqt_Sitosterol, output=vina_out_sdf_Sitosterol)

# ---- If poses collapse after 1-2 (as seen with Stigmasterol in this / other targets), uncomment ----
# STEROL_EXHAUSTIVENESS = 32
# STEROL_BOX_PAD = 6.0
# v_Sitosterol = Vina(sf_name='vina', seed=VINA_SEED)
# v_Sitosterol.set_receptor(RECEPTOR_PDBQT)
# v_Sitosterol.set_ligand_from_file(lig_pdbqt_Sitosterol)
# v_Sitosterol.compute_vina_maps(
#     center=[center['center_x'], center['center_y'], center['center_z']],
#     box_size=[size['size_x'] + STEROL_BOX_PAD,
#               size['size_y'] + STEROL_BOX_PAD,
#               size['size_z'] + STEROL_BOX_PAD])
# v_Sitosterol.dock(exhaustiveness=STEROL_EXHAUSTIVENESS, n_poses=10)
# v_Sitosterol.write_poses(vina_out_pdbqt_Sitosterol, n_poses=10, overwrite=True)
# energies_Sitosterol = v_Sitosterol.energies(n_poses=10)
# best_score_Sitosterol = round(float(energies_Sitosterol[0][0]), 3)
# pdbqt_to_sdf(pdbqt_file=vina_out_pdbqt_Sitosterol, output=vina_out_sdf_Sitosterol)


### 7.4. Total docking results table -- Lamium_amplexicaule vs PPAR-alpha (1K7L)

In [ ]:
docking_rows_amplexicaule_PPAR = [
    {"Compound": "Chrysoeriol",   "Target": "PPAR-alpha (1K7L)", "Best_Vina_Score_kcal_mol": best_score_Chrysoeriol},
    {"Compound": "Stigmasterol",  "Target": "PPAR-alpha (1K7L)", "Best_Vina_Score_kcal_mol": best_score_Stigmasterol},
    {"Compound": "GlcMethylKaempferol", "Target": "PPAR-alpha (1K7L)", "Best_Vina_Score_kcal_mol": best_score_GlcMethylKaempferol},
    {"Compound": "Sitosterol",          "Target": "PPAR-alpha (1K7L)", "Best_Vina_Score_kcal_mol": best_score_Sitosterol},
]

docking_df_amplexicaule_PPAR = pd.DataFrame(docking_rows_amplexicaule_PPAR).sort_values("Best_Vina_Score_kcal_mol").reset_index(drop=True)
docking_df_amplexicaule_PPAR.to_csv(os.path.join(DOCKING_DIR, "lamium_amplexicaule_vs_PPARalpha_vina.csv"), index=False)
docking_df_amplexicaule_PPAR


### 7.5. Figure -- docking score comparison, Lamium_amplexicaule vs PPAR-alpha

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
colors = ['#5a189a' if s == docking_df_amplexicaule_PPAR['Best_Vina_Score_kcal_mol'].min() else '#9d4edd'
          for s in docking_df_amplexicaule_PPAR['Best_Vina_Score_kcal_mol']]
ax.barh(docking_df_amplexicaule_PPAR['Compound'], docking_df_amplexicaule_PPAR['Best_Vina_Score_kcal_mol'], color=colors)
ax.set_xlabel('Best Vina docking score (kcal/mol)')
ax.set_title('Lamium amplexicaule compounds vs. PPAR-alpha (1K7L)')
ax.invert_yaxis()
for i, v in enumerate(docking_df_amplexicaule_PPAR['Best_Vina_Score_kcal_mol']):
    ax.text(v - 0.15, i, f"{v:.2f}", va='center', ha='right', fontsize=9, color='white')

# Reference line: fenofibrate's published PPAR-alpha docking score (-6.24 kcal/mol)
ax.axvline(-6.24, color='red', linestyle='--', linewidth=1, alpha=0.7, label='Fenofibrate (literature, -6.24)')
ax.legend(loc='lower right', fontsize=8)

plt.tight_layout()

fig_path_amplexicaule_PPAR = os.path.join(FIGURES_DIR, 'lamium_amplexicaule_vs_PPARalpha_scores.png')
plt.savefig(fig_path_amplexicaule_PPAR, dpi=300, bbox_inches='tight')
plt.show()
print("Saved figure:", fig_path_amplexicaule_PPAR)


### 7.6. ProLif interaction fingerprints -- Lamium_amplexicaule vs PPAR-alpha (each compound, own cell)

`n_jobs=1` avoids a multiprocessing/pickling crash in this environment;
`to_dataframe()` takes no arguments in ProLif 2.2.1. The 2D network is
saved as an interactive HTML file via `LigNetwork.save()` -- the original
notebook's `save_png(path)` call is not valid for this ProLif version
(`save_png()` takes no filename argument), so that call is replaced here.

In [ ]:
prot_fp = mda.Universe("1K7L_clean_H_fix.pdb")  # fix_protein() output
prot_fp = plf.Molecule.from_mda(prot_fp)


In [ ]:
# ---- Helper: render a ProLIF LigNetwork widget to a real PNG file ----
import os
import sys
import subprocess

_RENDER_WORKER_PATH = os.path.join(os.getcwd(), "_lignetwork_render_worker.py")

_RENDER_WORKER_SOURCE = '''
import sys
from playwright.sync_api import sync_playwright

html_path, png_path, width, height, wait_ms = sys.argv[1:6]
width, height, wait_ms = int(width), int(height), int(wait_ms)

with sync_playwright() as p:
    browser = p.chromium.launch()
    page = browser.new_page(viewport={"width": width, "height": height})
    page.goto(f"file://{html_path}")
    page.wait_for_timeout(wait_ms)
    try:
        page.evaluate("if (typeof network !== 'undefined') { network.fit({animation:false}); }")
        page.wait_for_timeout(300)
    except Exception:
        pass
    page.screenshot(path=png_path)
    browser.close()
'''

def lignetwork_to_png(net, png_path, width=1000, height=800, wait_ms=1500):
    """Render a ProLIF LigNetwork object to a standalone PNG file."""
    with open(_RENDER_WORKER_PATH, "w") as f:
        f.write(_RENDER_WORKER_SOURCE)

    tmp_html = png_path.rsplit(".", 1)[0] + "_tmp.html"
    net.save(tmp_html)

    result = subprocess.run(
        [sys.executable, _RENDER_WORKER_PATH, os.path.abspath(tmp_html),
         os.path.abspath(png_path), str(width), str(height), str(wait_ms)],
        capture_output=True, text=True,
    )

    if result.returncode != 0:
        print("Render worker STDOUT:", result.stdout)
        print("Render worker STDERR:", result.stderr)
        raise RuntimeError(f"lignetwork_to_png failed for {png_path}")

    os.remove(tmp_html)
    print("Saved PNG:", png_path)

In [ ]:
# ---- ProLif: Chrysoeriol ----
lig_suppl_Chrysoeriol = list(plf.sdf_supplier(vina_out_sdf_Chrysoeriol))
fp_Chrysoeriol = plf.Fingerprint()
fp_Chrysoeriol.run_from_iterable(lig_suppl_Chrysoeriol, prot_fp, n_jobs=1)
results_df_Chrysoeriol = fp_Chrysoeriol.to_dataframe()
print("=== Chrysoeriol ===")
display(results_df_Chrysoeriol)

valid_frames = results_df_Chrysoeriol[results_df_Chrysoeriol.any(axis=1)].index
if len(valid_frames) == 0:
    print("No interactions detected in any pose for Chrysoeriol - skipping network plot.")
else:
    frame_to_use = valid_frames[0]
    print("Using frame:", frame_to_use)
    net_Chrysoeriol = fp_Chrysoeriol.plot_lignetwork(lig_suppl_Chrysoeriol[0], kind="frame", frame=frame_to_use, rotation=270)
    net_png_Chrysoeriol = os.path.join(FIGURES_DIR, "Chrysoeriol_1K7L_2Dnetwork.png")
    lignetwork_to_png(net_Chrysoeriol, net_png_Chrysoeriol)

In [ ]:
# ---- ProLif: Stigmasterol ----
lig_suppl_Stigmasterol = list(plf.sdf_supplier(vina_out_sdf_Stigmasterol))
fp_Stigmasterol = plf.Fingerprint()
fp_Stigmasterol.run_from_iterable(lig_suppl_Stigmasterol, prot_fp, n_jobs=1)
results_df_Stigmasterol = fp_Stigmasterol.to_dataframe()
print("=== Stigmasterol ===")
display(results_df_Stigmasterol)

valid_frames = results_df_Stigmasterol[results_df_Stigmasterol.any(axis=1)].index
if len(valid_frames) == 0:
    print("No interactions detected in any pose for Stigmasterol - skipping network plot.")
else:
    frame_to_use = valid_frames[0]
    print("Using frame:", frame_to_use)
    net_Stigmasterol = fp_Stigmasterol.plot_lignetwork(lig_suppl_Stigmasterol[0], kind="frame", frame=frame_to_use, rotation=270)
    net_png_Stigmasterol = os.path.join(FIGURES_DIR, "Stigmasterol_1K7L_2Dnetwork.png")
    lignetwork_to_png(net_Stigmasterol, net_png_Stigmasterol)

In [ ]:
# ---- ProLif: GlcMethylKaempferol ----
lig_suppl_GlcMethylKaempferol = list(plf.sdf_supplier(vina_out_sdf_GlcMethylKaempferol))
fp_GlcMethylKaempferol = plf.Fingerprint()
fp_GlcMethylKaempferol.run_from_iterable(lig_suppl_GlcMethylKaempferol, prot_fp, n_jobs=1)
results_df_GlcMethylKaempferol = fp_GlcMethylKaempferol.to_dataframe()
print("=== GlcMethylKaempferol ===")
display(results_df_GlcMethylKaempferol)

valid_frames = results_df_GlcMethylKaempferol[results_df_GlcMethylKaempferol.any(axis=1)].index
if len(valid_frames) == 0:
    print("No interactions detected in any pose for GlcMethylKaempferol - skipping network plot.")
else:
    frame_to_use = valid_frames[0]
    print("Using frame:", frame_to_use)
    net_GlcMethylKaempferol = fp_GlcMethylKaempferol.plot_lignetwork(lig_suppl_GlcMethylKaempferol[0], kind="frame", frame=frame_to_use, rotation=270)
    net_png_GlcMethylKaempferol = os.path.join(FIGURES_DIR, "GlcMethylKaempferol_1K7L_2Dnetwork.png")
    lignetwork_to_png(net_GlcMethylKaempferol, net_png_GlcMethylKaempferol)

In [ ]:
# ---- ProLif: Sitosterol ----
lig_suppl_Sitosterol = list(plf.sdf_supplier(vina_out_sdf_Sitosterol))
fp_Sitosterol = plf.Fingerprint()
fp_Sitosterol.run_from_iterable(lig_suppl_Sitosterol, prot_fp, n_jobs=1)
results_df_Sitosterol = fp_Sitosterol.to_dataframe()
print("=== Sitosterol ===")
display(results_df_Sitosterol)

valid_frames = results_df_Sitosterol[results_df_Sitosterol.any(axis=1)].index
if len(valid_frames) == 0:
    print("No interactions detected in any pose for Sitosterol - skipping network plot.")
else:
    frame_to_use = valid_frames[0]
    print("Using frame:", frame_to_use)
    net_Sitosterol = fp_Sitosterol.plot_lignetwork(lig_suppl_Sitosterol[0], kind="frame", frame=frame_to_use, rotation=270)
    net_png_Sitosterol = os.path.join(FIGURES_DIR, "Sitosterol_1K7L_2Dnetwork.png")
    lignetwork_to_png(net_Sitosterol, net_png_Sitosterol)

### 7.7. Figure -- best-scoring compound vs. the real native ligand (GW409544)

In [ ]:
best_row_amplexicaule_PPAR = docking_df_amplexicaule_PPAR.iloc[0]
best_compound_amplexicaule_PPAR = best_row_amplexicaule_PPAR['Compound']
print("Best-scoring compound:", best_compound_amplexicaule_PPAR, "|", best_row_amplexicaule_PPAR['Best_Vina_Score_kcal_mol'], "kcal/mol")

best_sdf_amplexicaule_PPAR = {
    "Chrysoeriol":  vina_out_sdf_Chrysoeriol,
    "Stigmasterol": vina_out_sdf_Stigmasterol,
    "GlcMethylKaempferol": vina_out_sdf_GlcMethylKaempferol,
    "Sitosterol":          vina_out_sdf_Sitosterol,
}[best_compound_amplexicaule_PPAR]

view = py3Dmol.view()
view.removeAllModels()
view.setViewStyle({'style':'outline','color':'black','width':0.1})

view.addModel(open(RECEPTOR_PDB,'r').read(), format='pdb')
Prot = view.getModel()
Prot.setStyle({'cartoon':{'arrows':True, 'tubes':True, 'style':'oval', 'color':'white'}})
view.addSurface(py3Dmol.VDW, {'opacity':0.6,'color':'white'})

view.addModel(open(NATIVE_LIG,'r').read(), format='mol2')
ref_m = view.getModel()
ref_m.setStyle({}, {'stick':{'colorscheme':'magentaCarbon','radius':0.2}})

results_amp_PPAR = Chem.SDMolSupplier(best_sdf_amplexicaule_PPAR)
p_amp_PPAR = Chem.MolToMolBlock(results_amp_PPAR[0], False)
print('Reference (GW409544): Magenta | Best compound pose: Cyan')
print('Pose: {} | Score: {}'.format(results_amp_PPAR[0].GetProp('Pose'), results_amp_PPAR[0].GetProp('Score')))

view.addModel(p_amp_PPAR, 'mol')
x = view.getModel()
x.setStyle({}, {'stick':{'colorscheme':'cyanCarbon','radius':0.2}})

view.zoomTo()
view.show()
